# 🧪 Guia de referência: Teste A/B

Guia passo a passo para **testes A/B**, do planejamento à produção, com os 4 métodos do curso de Teste A/B da Comunidade DS: **A/B frequentista, A/B/n, A/B Bayesiano e Multi-Armed Bandit (MAB)**. Os exemplos usam os números das aulas e o projeto de Segmentação de clientes do portfólio. Para começar um teste novo, use o arquivo **`Estrutura_Teste_AB.ipynb`**, que tem os mesmos títulos sem as explicações.

> 🧭 **Como usar este guia**
> - As fases seguem o roteiro CRISP-DM do portfólio, e dentro delas os **5 passos do curso**: escolha do método → design → coleta e preparação → teste das hipóteses → conclusões.
> - Cada etapa tem: **📖 O que é**, **🧠 Conceitos** (com exemplo numérico), **✅ O que fazer**, **⚠️ Erros comuns** e as aulas de referência entre parênteses.
> - Está com pressa? Comece pela figura **"Qual método usar?"** (Fase 01, item 1) e pelo **mapa de escolha do teste** (Fase 03, item 1).
> - Ficou com dúvida sobre um termo? Veja o **📚 Glossário** no fim do arquivo.
> - Regra de ouro das anotações: **todo número citado numa nota precisa aparecer na saída executada** da célula acima.

# FASE 01: ENTENDIMENTO DO NEGÓCIO

📖 **O que é:** antes de dividir qualquer visitante em grupos, entender **qual decisão o teste vai apoiar**. Quase nunca alguém pede "um teste A/B": o Product Manager chega com "fiz uma página nova, ela é melhor que a atual?" (Aula 2). Transformar essa pergunta num experimento é o trabalho desta fase.

## 🏢 Contexto

📖 Descreva **a mudança que será testada**, quem pediu e quanto está em jogo. Exemplo da Aula 31: a página A vende um produto de R$ 4.500, recebe 1 milhão de visitas por dia e converte 13%. Trocar pela página B **sem testar** é arriscado: se a conversão cair para 12%, a perda é de **R$ 45 milhões por dia**.

✅ Inclua: a versão atual (controle), a versão nova (tratamento), o **stakeholder** (quem decide com o resultado) e o motivo da mudança.

🧠 **Por que não basta trocar a página e comparar o antes e o depois?** Porque o público muda de um dia para o outro (feriado, campanha, clima). A diferença observada pode vir do **público**, e não da página (Aulas 3 e 31). Só o **sorteio simultâneo** dos grupos isola o efeito da mudança.

## 🎯 Objetivo

📖 Uma frase com **o que será comparado**, **por qual métrica** e **para quê**: "descobrir se a página nova aumenta a conversão de compra, para decidir se ela substitui a atual".

## 🧩 Formato da solução

📖 O que será entregue: a **decisão** (implantar, manter ou testar de novo), o **ganho esperado em R$** com intervalo, e, quando o teste vira rotina, um **processo automático** (API ou job) que aloca o tráfego e atualiza os resultados (Aulas 44 e 45).

## 🗺️ Planejamento da solução (os 5 passos do teste)

📖 O curso repete o mesmo roteiro em todos os ciclos (Aulas 2, 15, 19 e 33). Ele é mais curto que o de Machine Learning, mas cada passo tem armadilhas próprias:

![Os 5 passos](imagens/ab_cinco_passos.svg)

| Passo | Pergunta que responde | Onde está neste guia |
|---|---|---|
| **1. Escolha do método** | A/B, A/B/n, Bayesiano ou MAB? | Fase 01, item 1 |
| **2. Design do experimento** | Hipóteses, métrica, α, poder, efeito mínimo, amostra, grupos | Fase 01, item 2 |
| **3. Coleta e preparação dos dados** | O sorteio funcionou? Há duplicados? | Fase 02 |
| **4. Teste das hipóteses** | Qual teste estatístico e o que ele diz | Fase 03 |
| **5. Conclusões** | O que fazer, quanto se ganha, qual método foi o ideal | Fase 04 |

⚠️ **A/B é o formato, não o teste estatístico.** "Teste A/B" diz **como dividir** os grupos. Quem decide se a diferença é real é o **método de inferência** (Z, qui-quadrado, t, Mann-Whitney, Beta…), escolhido pelo **tipo do dado** (Aula 19).

## 1 - ESCOLHA DO MÉTODO

📖 **O que é:** decidir **qual família de teste** responde à pergunta do negócio. São 4 métodos, um por ciclo do curso:

| Método | Em uma frase | Quando brilha |
|---|---|---|
| **A/B frequentista** (Ciclo 01) | 2 versões, amostra calculada antes, decisão no fim pelo p-valor | Decisão única e importante, com tempo para esperar e necessidade de um p-valor "oficial" |
| **A/B/n** (Ciclo 02) | O mesmo, com 3 ou mais versões e correção para várias comparações | Várias ideias de uma vez (4 ofertas, 5 páginas) sem fazer testes A/B em sequência |
| **A/B Bayesiano** (Ciclo 03) | Calcula **P(B > A)** e o **risco** de escolher B, atualizados a cada dia | Quando o negócio não quer esperar o tamanho de amostra completo, ou quer a resposta em "chance de B ser melhor" |
| **MAB, Multi-Armed Bandit** (Ciclo 04) | Desloca o tráfego para a melhor versão **enquanto aprende** | Quando mandar gente para a versão pior custa caro, ou o teste se repete sempre (anúncios, recomendações, campanhas) |

![Qual método usar](imagens/ab_qual_metodo.svg)

✅ **O que fazer**
- Responder às 3 perguntas da figura: quantas versões? É preciso decidir antes do fim? Perder conversão na versão pior custa caro?
- Registrar a escolha e o motivo no notebook, antes de olhar os dados.

💡 Os métodos **se combinam**. Um caminho comum, e o usado no portfólio, é validar a regra uma vez com um **A/B/n** (lido também pelo lado Bayesiano) e, no dia a dia, alocar o tráfego com um **MAB** que tem um **grupo de controle fixo**. A Fase 04 mostra como escolher.

## 2 - DESIGN DO EXPERIMENTO

📖 **O que é:** combinar **antes** de coletar os dados tudo o que define o teste: hipóteses, métrica, nível de significância, poder, efeito mínimo, tamanho da amostra e como os grupos serão separados (Aulas 5, 6, 7 e 19). Mudar qualquer um desses itens depois de ver o resultado invalida o teste.

### 2.1. Formulação das hipóteses

🧠 **Conceitos**
- **Hipótese nula (H0):** o "status quo", ou seja, **não houve efeito**: a diferença entre A e B é só fruto do sorteio (Aula 31).
- **Hipótese alternativa (H1):** **houve efeito**: a diferença vem da mudança.
- **Teste bicaudal:** H1 diz "B é **diferente** de A" (pode ser melhor ou pior). Os 5% de α se dividem entre as duas caudas, 2,5% de cada lado.
- **Teste unicaudal:** H1 diz "B é **maior** que A". Os 5% ficam numa cauda só. Só use quando o efeito **só puder** acontecer numa direção (Aula 7).

![Unicaudal × bicaudal](imagens/ab_uni_bicaudal.svg)

✅ **Exemplo (Aula 7):** H0: p = p0 e H1: p ≠ p0 (bicaudal), ou H1: p > p0 (unicaudal).

⚠️ **Na dúvida, use o bicaudal.** O unicaudal "facilita" achar significância e esconde o caso em que B é **pior** que A, justamente o que mais importa evitar.

### 2.2. Escolha da métrica (variável)

📖 A métrica decide **qual ramo do mapa de testes** será usado (Fase 03, item 1).

| Tipo de métrica | Exemplos | Ramo do mapa |
|---|---|---|
| **Proporção (sim/não)** | Conversão, CTR (cliques ÷ visitantes), resposta à campanha | Categórico → Z ou qui-quadrado |
| **Média de um valor contínuo** | Ticket médio, lucro por cliente, tempo na página | Contínuo → t, Welch, Mann-Whitney, ANOVA, Kruskal-Wallis |

✅ **O que fazer**
- Perguntar ao negócio **como ele calcula a taxa hoje**. O CTR pode ser cliques ÷ visitantes ou cliques ÷ cliques na página (Aula 19), e cada definição muda o resultado.
- Ter **uma métrica principal**, a que decide, e no máximo algumas de **proteção** (que não podem piorar, como cancelamentos e reclamações).
- Sempre que possível, levar a métrica para **dinheiro**: conversão × ticket × margem. Uma conversão maior com ticket menor pode dar prejuízo.

### 2.3. Nível de confiança, nível de significância e poder

🧠 **Nível de confiança (95%)** — Aula 6

O erro mais comum é dizer "estou 95% certo de que a minha amostra representa a população". O certo é: **se eu repetir a amostragem 20 vezes e calcular 20 intervalos, espero que 19 deles contenham o valor real** (19 ÷ 20 = 95%), mesmo sem saber qual é esse valor.

![Nível de confiança](imagens/ab_intervalos_confianca.svg)

🧠 **Nível de significância (α = 5%)** é o outro lado: a chance de **1 intervalo em 20** errar. É o **risco máximo** de rejeitar H0 quando ela é verdadeira, ou seja, de ver um efeito que não existe (erro tipo I). A anotação da Aula 6 resume com um exemplo: "olhar para uma pessoa acima do peso e dizer que ela está grávida".

🧠 **Poder (1 − β = 80%)**: a chance de o teste **enxergar um efeito que existe de verdade**. Os 20% que sobram (β) são o erro tipo II: dizer que não há efeito quando há.

![Erros e poder](imagens/ab_erros_poder.svg)

| | H0 é verdade (não há efeito) | H1 é verdade (há efeito) |
|---|---|---|
| **O teste rejeita H0** | ❌ Erro tipo I (α): alarme falso | ✅ Acerto (poder) |
| **O teste não rejeita H0** | ✅ Acerto | ❌ Erro tipo II (β): deixou passar |

🧠 **O p-valor mede a aleatoriedade.** A Aula 3 dá a intuição: assumimos H0 como verdade e calculamos a chance de ver uma diferença **tão grande ou maior** só por sorte. Se essa chance (o p-valor) for **menor** que o máximo de aleatoriedade que aceitamos (α), atribuímos a diferença à mudança. Se for maior, há aleatoriedade demais para afirmar qualquer coisa.

⚠️ "Não rejeitar H0" **não é** provar que A e B são iguais: pode ser só falta de amostra.

### 2.4. Tamanho do efeito (effect size)

📖 **O que é:** a **menor diferença que vale a pena enxergar**. Ele define o tamanho da amostra: diferenças pequenas exigem muitos dados; diferenças grandes, poucos (Aula 6).

🧠 **Duas maneiras de definir o efeito mínimo (Aula 8)**
1. **Meta do negócio:** "o time espera um aumento de 2% na conversão".
2. **Custo da mudança (ROI):** o menor efeito que **paga** a implementação. Exemplo da Aula 8: um botão novo custa 150 h × R$ 500 = **R$ 75 mil**. Com lucro de R$ 3 por venda, ele precisa trazer 75.000 ÷ 3 = **25 mil vendas a mais por ano**. A página vende 730 mil por ano, então o aumento mínimo é 25.000 ÷ 730.000 = **3,42%**.

🧠 **Como medir o efeito (Aula 6)**

| Medida | Para quê | Pequeno / médio / grande | Python |
|---|---|---|---|
| **d de Cohen** | Diferença de **médias** em desvios padrão | 0,2 / 0,5 / 0,8 | `pingouin.compute_effsize(x, y, eftype='cohen')` |
| **Delta de Glass** | Como o d, mas usa só o desvio do **controle** (quando o tratamento muda a variabilidade) | 0,2 / 0,5 / 0,8 | `eftype='glass'` |
| **g de Hedges** | O d corrigido para **amostras pequenas** (n < 20) | 0,2 / 0,5 / 0,8 | `eftype='hedges'` |
| **h de Cohen** | Diferença entre duas **proporções** (conversão) | 0,2 / 0,5 / 0,8 | `proportion_effectsize(p1, p2)` |
| **w de Cohen** | Diferença entre **3 ou mais** proporções (qui-quadrado, A/B/n) | 0,1 / 0,3 / 0,5 | calculado da tabela de proporções esperadas (Fase 03, item 3) |

💡 O h de Cohen é h = 2 × (arcsen √p1 − arcsen √p2). A raiz e o arco-seno "esticam" as proporções perto de 0% e 100%, onde uma diferença de 1 ponto pesa mais.

### 2.5. Tamanho da amostra e duração do teste

📖 Com α, poder e efeito mínimo definidos, o tamanho da amostra **sai de uma conta**, não do "acho que mil é suficiente".

```python
import math
from statsmodels.stats.proportion import proportion_effectsize
import statsmodels.stats.power as sms

efeito = proportion_effectsize(0.13, 0.15)                        # h de Cohen: de 13% para 15%
n_por_grupo = sms.NormalIndPower().solve_power(efeito, power=0.80, alpha=0.05)
math.ceil(n_por_grupo)                                            # 4.720 por grupo (PA James)
```

![Tamanho da amostra × efeito](imagens/ab_tamanho_amostra.svg)

🔎 Partindo de 13% de conversão: enxergar **+2 pontos** (15%) exige **4.720** visitantes por grupo; **+1 ponto** (14%), **18.326**; **+7 pontos** (20%), só **437**. Reduzir o efeito pela metade multiplica a amostra por cerca de 4.

✅ **O que fazer**
- Calcular o **n por grupo** e converter em **dias**: n ÷ visitas por dia em cada grupo. Rodar pelo menos **1 semana cheia** (o comportamento muda entre os dias da semana).
- Para médias (ticket, lucro), usar o d de Cohen com `TTestIndPower`. Para 3 ou mais grupos, o w de Cohen com `GofChisquarePower` (Fase 03, item 3).
- Se a amostra necessária for inviável, **aumentar o efeito mínimo** (testar mudanças maiores) ou trocar de método (Bayesiano ou MAB).

### 2.6. Separação dos grupos

📖 **O que é:** sortear quem vê cada versão. O sorteio deixa os grupos **parecidos em tudo**, e a única diferença sistemática passa a ser a versão (Aula 3).

![O sorteio](imagens/ab_sorteio.svg)

✅ **O que fazer**
- **Unidade do sorteio:** o usuário (não a visita), para a mesma pessoa não ver as duas versões.
- **Proporção:** 50/50 no A/B; partes iguais no A/B/n. O MAB muda a proporção sozinho.
- **Flag do teste:** uma coluna que registra o grupo de cada usuário (Aula 19).
- **Só uma coisa muda** entre os grupos (Aula 15): se a página nova tem outra cor **e** outro texto, não dá para saber qual dos dois causou o efeito.

# FASE 02: PREPARAÇÃO DOS DADOS

📖 **O que é:** o passo 3 do curso, **coletar e preparar os dados**. Um teste com dados sujos dá uma resposta precisa para a pergunta errada.

## 1 - COLETA DOS DADOS

📖 Registrar, para cada usuário: **id, grupo (flag), data e hora, se converteu e o valor**. Ferramentas como o **Google Analytics** registram os eventos da jornada do cliente na página (Aula 16).

✅ **O que fazer**
- Guardar o **log bruto** (um evento por linha) e montar as tabelas a partir dele no notebook.
- Conferir se a coleta começou **ao mesmo tempo** nos dois grupos.

## 2 - LIMPEZA E CONFERÊNCIAS

✅ **O checklist do curso (Aula 7 e PA Bond), mais um passo**
1. **Usuários duplicados ou nos dois grupos:** remover. No PA James, parte dos usuários do controle viu a página nova, um erro de implementação.
2. **Grupo × versão:** o grupo controle precisa ter visto **só** a versão A.
3. **Divisão do sorteio (SRM, *sample ratio mismatch*):** com 50/50 planejado, os grupos precisam ter tamanhos próximos. Um **qui-quadrado** com os tamanhos observados contra os esperados avisa se o sorteio quebrou (p < 0,01 = investigar antes de seguir).
4. **Grupos parecidos (homogeneidade):** comparar o perfil (renda, idade, canal) dos grupos. Diferenças grandes indicam um sorteio com problema.
5. **Tabela de conversões:** por grupo, **convertidos e não convertidos** (não o total!) — é ela que entra no qui-quadrado (Aula 19, passo 3.3).

```python
from scipy import stats
tamanhos = [5012, 4876]                                     # usuários observados em A e B
p_srm = stats.chisquare(tamanhos, f_exp=[sum(tamanhos) / 2] * 2).pvalue
```

⚠️ **Erros comuns**
- Montar a tabela do qui-quadrado com **convertidos e total** em vez de convertidos e **não convertidos**.
- Remover outliers **só de um grupo**, ou depois de ver qual remoção "ajuda" o resultado.

## 3 - ANÁLISE EXPLORATÓRIA

✅ **O que fazer**
- Conversão **por dia** em cada grupo: um pico isolado (uma campanha, um bug) pode decidir o teste sozinho.
- Distribuição da métrica contínua (ticket, lucro): muito assimétrica ou cheia de zeros pede Mann-Whitney ou transformação (log).
- Conferir as **suposições paramétricas** (Fase 03, item 1) antes de escolher o teste.

# FASE 03: TESTE DAS HIPÓTESES

📖 **O que é:** o passo 4 do curso. Primeiro escolher o **teste estatístico** pelo tipo de dado (o mapa da Aula 9) e depois aplicá-lo **no formato** escolhido: A/B, A/B/n, Bayesiano ou MAB.

## 1 - Como escolher o teste estatístico (o mapa da Aula 9)

📖 O mapa da Aula 9 escolhe o teste com **4 perguntas**: qual o tipo de dado? Relação ou diferença? Diferença de quê? Quantos grupos? E, no fim, as suposições paramétricas são satisfeitas?

![Mapa de escolha do teste](imagens/ab_mapa_testes.svg)

🧠 **Suposições paramétricas** (as 3 condições dos testes t e ANOVA)
1. **Independência:** um grupo não influencia o outro.
2. **Normalidade:** os dados seguem uma curva normal (teste de **Shapiro-Wilk**).
3. **Variâncias iguais** (teste de **Levene**, ou Fmax e Bartlett, que aparecem no mapa).

Se uma delas falhar, "o mundo não acaba" (anotação da Aula 9): **transforme** os dados (log) e confira de novo, ou use a versão **sem suposições** (Mann-Whitney, Kruskal-Wallis).

🧠 **Todos os testes do mapa**

| Situação | Teste | Python | Exemplo do curso |
|---|---|---|---|
| Proporção de 1 grupo × um valor de referência | **Z de 1 proporção** | `proportions_ztest(cont, n, value=p0)` | Exercício extra: mais de 20% de peças com defeito? |
| Proporções de **2 grupos** | **Z de 2 proporções** | `proportions_ztest([cA, cB], [nA, nB])` | Aula 22: conversão de 2 páginas de venda |
| Contagens de **2 ou mais grupos** | **Qui-quadrado de independência** | `stats.chi2_contingency(tabela)` | Aula 22: gênero × preferência partidária |
| Uma categórica segue a distribuição esperada? | **Qui-quadrado de aderência** (*goodness of fit*) | `stats.chisquare(obs, f_exp=esp)` | Aula 22: reclamações iguais em todos os dias da semana? |
| Contagens pequenas (célula com menos de 5) | **Exato de Fisher** | `stats.fisher_exact(tabela_2x2)` | — |
| Média de 1 grupo × um valor | **t de 1 amostra** | `stats.ttest_1samp(x, valor)` | Aula 10: a altura média das plantas é 15 cm? |
| Médias de 2 grupos, suposições OK | **t de 2 amostras** | `stats.ttest_ind(a, b)` | Aula 10: duas espécies têm a mesma altura? |
| Médias de 2 grupos, variâncias diferentes | **Welch** | `stats.ttest_ind(a, b, equal_var=False)` | PA Bond: gasto médio por grupo |
| 2 grupos sem suposições (assimetria, outliers) | **Mann-Whitney U** | `stats.mannwhitneyu(a, b)` | Aula 10, solução 2 |
| Mesma unidade medida 2 vezes (antes × depois) | **Wilcoxon pareado** | `stats.wilcoxon(antes, depois)` | — |
| Médias de 3 ou mais grupos, suposições OK | **ANOVA de 1 fator** (+ **Tukey** para saber qual) | `stats.f_oneway(*grupos)` | Aula 11: 4 tratamentos; exercício 9: 3 métodos de estudo |
| Dois fatores ao mesmo tempo (e a interação) | **ANOVA de 2 fatores** | `statsmodels` `ols` + `anova_lm` | Exercício 10: formato × investimento no CTR |
| 3 ou mais grupos sem suposições | **Kruskal-Wallis** (+ **Dunn**) | `stats.kruskal(*grupos)` | Aula 11, solução 2 |
| Variâncias iguais? | **Levene, Bartlett ou Fmax** | `stats.levene(a, b)` | Mapa da Aula 9 |
| Relação entre variáveis | **Correlação ou regressão** | `stats.spearmanr`, `statsmodels` | Mapa da Aula 9 |

🧠 **Z ou t?** O **Z** compara **proporções** (e médias quando o desvio padrão da população é conhecido). O **t** compara **médias** com o desvio estimado da amostra (Aula 22). Para conversão, use o Z de 2 proporções ou o qui-quadrado: com 2 grupos, os dois dão **o mesmo p-valor** (o qui-quadrado é o Z ao quadrado).

🧠 **Mann-Whitney × Welch (anotação da Aula 11):** o Mann-Whitney é o padrão quando as condições paramétricas não são satisfeitas; o Welch é mais conservador e oscila mais com o tamanho da amostra. Rodar os dois e ver se concordam é uma boa conferência.

✅ **Treino: os 10 exercícios da Aula 23 e o teste de cada um (Aulas 24 e 25)**

| # | Problema | Tipo de dado | Teste |
|---|---|---|---|
| 1 | CTR do anúncio 1 (20 de 1.000) × anúncio 2 (30 de 900) | Proporção, 2 grupos | Z de 2 proporções (ou qui-quadrado) |
| 2 | Cartas: 30% novatos, 60% veteranos, 10% All-Stars? | Contagens × distribuição afirmada | Qui-quadrado de aderência |
| 3 | Preferência de voto difere entre homens e mulheres? | 2 categóricas | Qui-quadrado de independência |
| 4 | As meninas tiram, em média, mais de 600 (σ = 100 conhecido)? | Média de 1 grupo × valor | Z de 1 amostra (σ conhecido) ou t de 1 amostra |
| 5 | A farmácia entrega em menos de 120 min (σ = 30, n = 49)? | Média × valor, σ conhecido | Z de 1 amostra, unicaudal |
| 6 | Linha A (18 defeitos em 200) × linha B (25 em 600) | Proporção, 2 grupos | Z de 2 proporções |
| 7 | Estado civil está associado ao nível de educação? | 2 categóricas | Qui-quadrado de independência |
| 8 | Visitas iguais todos os dias da semana? | Contagens × distribuição uniforme | Qui-quadrado de aderência |
| 9 | 3 métodos de estudo dão notas médias diferentes? | Média, 3 grupos | ANOVA de 1 fator (ou Kruskal-Wallis) |
| 10 | Formato e investimento do anúncio influenciam o CTR? | Média, 2 fatores | ANOVA de 2 fatores |

🔎 Exercício 1 resolvido: Z = −1,81 e **p = 0,070**. Com α = 5%, **não** há evidência de que os CTRs (2,0% × 3,3%) sejam diferentes: a diferença parece grande, mas a amostra é pequena para ela.

## 2 - Ciclo 01: Teste A/B frequentista

📖 **O que é:** duas versões, amostra calculada **antes**, decisão **no fim** pelo p-valor. É o método mais usado para validar se uma página, cupom ou preço é melhor que o atual (Aula 0).

✅ **O roteiro**
1. Hipóteses bicaudais, α = 5%, poder = 80%, efeito mínimo (Fase 01, item 2).
2. Tamanho da amostra com o **h de Cohen** e `NormalIndPower`.
3. Conferências da Fase 02 (duplicados, SRM, grupos parecidos).
4. **Taxa:** Z de 2 proporções ou qui-quadrado, mais o **intervalo de confiança da diferença**.
5. **Valor (ticket, lucro):** Welch e Mann-Whitney.
6. **Conclusão em R$:** diferença de conversão × visitas × ticket, com o intervalo.

```python
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

# Aula 22: página atual 15 vendas em 1.150 visitas (1,3%); página nova 25 em 1.200 (2,1%)
z, p = proportions_ztest([25, 15], [1200, 1150])
ic_baixo, ic_alto = confint_proportions_2indep(25, 1200, 15, 1150)   # IC 95% da diferença (B − A)
```

🔎 Resultado: **p = 0,144**, e o IC 95% da diferença vai de **−0,3 a +1,9 pontos**, ou seja, inclui o zero. Com α = 5%, **não** há evidência de que a página nova converta mais: com só 40 vendas no total, a amostra é pequena para uma diferença de 0,8 ponto.

⚠️ **Erros comuns**
- **Espiar e parar** quando o p-valor cai abaixo de 5% (Ciclo 03 mostra o tamanho do problema).
- Ler "p > 0,05" como "as versões são iguais".
- Esquecer o **IC**: o p-valor diz se há efeito; o intervalo diz **quanto**, o que o negócio precisa saber.

## 3 - Ciclo 02: Teste A/B/n

📖 **O que é:** o mesmo formato do A/B, com um **controle** e **várias variações** ao mesmo tempo (Aulas 14 e 17). Exemplo do curso: 4 ofertas (vitalícia, anual, semestral, mensal) contra a compra avulsa, sem tempo para 4 testes A/B em sequência.

🧠 **O problema das várias comparações: o erro α acumulado (Aula 17)**

Cada comparação tem 5% de chance de alarme falso. Com várias, a chance de **pelo menos um** alarme falso cresce: **α\* = 1 − (1 − α)ⁿ**.

![Erro α acumulado](imagens/ab_alfa_acumulado.svg)

🔎 Com **6 comparações**, a chance de um falso vencedor já é de **26%**; com **10** (4 variantes + controle, todos os pares), **40%**.

✅ **Por isso o A/B/n tem duas etapas**
1. **Teste geral:** "existe **alguma** diferença?" Para taxas, o **qui-quadrado** com a tabela k × 2 (convertidos e não convertidos). Para médias, **ANOVA** ou **Kruskal-Wallis**.
2. **Post-hoc** (só se o geral der significativo): "**onde** está a diferença?", com as comparações par a par **corrigidas**.

| Correção | Como funciona | Quando usar |
|---|---|---|
| **Bonferroni** | Multiplica cada p-valor pelo número de comparações k (ou compara com α ÷ k) | Simples e conservadora; o padrão do curso |
| **Šidák** | Compara com 1 − (1 − α)^(1/k) | Um pouco menos conservadora que a Bonferroni |
| **FDR (Benjamini-Hochberg)** | Controla a fração de falsos positivos entre os "vencedores" | Muitas comparações, quando alguns falsos positivos são aceitáveis |
| **Tukey HSD** | Post-hoc da ANOVA (médias) | Depois de uma ANOVA significativa |
| **Dunn** | Post-hoc do Kruskal-Wallis | Depois de um Kruskal-Wallis significativo |

```python
from itertools import combinations
from scipy import stats
from statsmodels.stats.multitest import multipletests

tabela = [[conv, total - conv] for conv, total in zip(convertidos, visitantes)]   # k x 2
p_geral = stats.chi2_contingency(tabela)[1]                                        # existe diferença?

pares = list(combinations(range(len(tabela)), 2))
p_pares = [stats.chi2_contingency([tabela[i], tabela[j]])[1] for i, j in pares]
rejeita, p_corrigido, _, _ = multipletests(p_pares, alpha=0.05, method='bonferroni')
```

🧠 **Tamanho da amostra no A/B/n: o w de Cohen (Aula 19)**

```python
import numpy as np
import statsmodels.stats.power as sms

taxas = np.array([0.10, 0.12, 0.14, 0.10])                  # taxas esperadas de cada versão
k = len(taxas)
observado = np.column_stack([taxas, 1 - taxas]) / k          # tabela k x 2 em proporções
esperado = np.outer(observado.sum(axis=1), observado.sum(axis=0))
w = np.sqrt(((observado - esperado) ** 2 / esperado).sum())  # w de Cohen
n_total = sms.GofChisquarePower().solve_power(effect_size=w, alpha=0.05, power=0.80, n_bins=k)
n_por_grupo = int(np.ceil(n_total / k))
```

🔎 Com essas 4 taxas, w = **0,052** e são precisos **1.009** visitantes por versão.

⚠️ **Use `GofChisquarePower`, não `TTestIndPower`.** O notebook *PA James ABN* do curso calcula a amostra do qui-quadrado com `TTestIndPower` (a fórmula do teste t), o que dá **5.811** por versão no exemplo acima, quase 6 vezes mais. No projeto de Segmentação do portfólio, uma simulação de 4.000 experimentos confirmou que o `GofChisquarePower` entrega o poder planejado (79,6% com 134 contatos, para 80% planejados).

## 4 - Ciclo 03: Teste A/B Bayesiano

📖 **O que é:** em vez de perguntar "se não houvesse diferença, qual a chance de ver isso?" (p-valor), pergunta **"qual a probabilidade de B ser melhor que A?"** e **"quanto eu perco se escolher B e estiver errado?"**. O problema da Aula 27: o diretor de produto **não quer esperar 30 dias** pelo tamanho de amostra completo.

🧠 **Por que não basta olhar o p-valor todo dia?** Porque ele **oscila**. Recalculando o p-valor a cada dia com os dados acumulados (Aula 30), ele pode cruzar 5% por acaso e voltar a subir:

![Espiar o p-valor](imagens/ab_espiar_pvalor.svg)

🔎 Na figura, A e B são **idênticas** (as duas convertem 10%) e mesmo assim o p-valor caiu abaixo de 0,05 no **dia 10**. Em 2.000 testes A/A simulados, quem espia todo dia por 30 dias declara um falso vencedor em **27,2%** das vezes; quem olha só no fim, em **5,2%** (os 5% prometidos pelo α).

| | **Frequentista** | **Bayesiano** |
|---|---|---|
| Pergunta | "Se não houvesse diferença, qual a chance de ver um resultado assim?" | "Qual a chance de B ser melhor? Quanto perco se errar?" |
| Quando decide | Só no fim, com a amostra planejada | A qualquer momento, pela regra de parada combinada antes |
| Resultado para o negócio | "Rejeitamos H0 com 5% de significância" | "B tem 96% de chance de ser melhor e o risco é de 0,1 ponto" |
| Ponto fraco | Espiar invalida o α | É preciso escolher a priori e a perda máxima |

🧠 **Como funciona: a distribuição Beta**

A conversão de cada versão é descrita por uma **Beta(1 + conversões, 1 + não conversões)**, a distribuição natural para "sucesso ou fracasso" (Aula 33):
- **A priori:** antes dos dados, Beta(1, 1), em que qualquer taxa entre 0% e 100% é igualmente possível.
- **Os dados** (quantos converteram em quantos) seguem uma **Binomial**, que é a **verossimilhança**.
- **A posteriori:** a priori atualizada pelos dados, de novo uma Beta. A cada dia, basta somar as novas conversões.

![A Beta aprendendo](imagens/ab_beta_posteriori.svg)

🧠 **As três contas, por Monte Carlo** (sortear milhares de valores e contar; Aulas 31 e 32)
- **P(B > A):** em quantos sorteios a taxa de B superou a de A.
- **Perda esperada de escolher B:** a média de max(A − B, 0), ou seja, quanto de conversão se perde, em média, se B for pior. É o **risco** da decisão.
- **Regra de parada (Aulas 33 e 35):** escolher B quando a **perda esperada de B ≤ perda máxima aceitável**, definida no design (passo 2.3 da Aula 33).

```python
import numpy as np
rng = np.random.default_rng(42)

# Aula 35: A = 234 cliques em 2.488 visitas; B = 222 cliques em 2.209 visitas
taxa_a = rng.beta(1 + 234, 1 + 2488 - 234, 200_000)
taxa_b = rng.beta(1 + 222, 1 + 2209 - 222, 200_000)

p_b_melhor = (taxa_b > taxa_a).mean()
perda_b = np.maximum(taxa_a - taxa_b, 0).mean()             # risco de escolher B
decide_b = perda_b <= 0.0001                                 # perda máxima do curso
```

🔎 Resultado: P(B > A) = **77%** e perda esperada de B = **0,11 ponto** (0,00115). Com a perda máxima do curso, de **0,0001**, ainda **não** dá para decidir: é preciso mais dados. O frequentista, com os mesmos números, dá p = **0,456**.

![Perda esperada e regra de parada](imagens/ab_perda_esperada.svg)

🔎 Na simulação da figura (A = 10%, B = 12%, 300 visitas por dia em cada versão), a perda esperada de B ficou abaixo do limite de 0,1 ponto no **dia 5**, quando P(B > A) era de **96,3%**.

✅ **O que fazer**
- Definir a **perda máxima aceitável** no design, pensando no custo do erro. Ela **depende do tamanho do negócio**: 0,0001 faz sentido para um site com dezenas de milhares de visitas por dia em cada página; para uma campanha com poucas dezenas de contatos por dia, um limite assim nunca seria atingido.
- Reportar P(B > A), a perda esperada e, se houver um custo, **P(dá lucro)** = chance de a taxa passar do ponto de equilíbrio.
- Sortear **direto da Beta** (`rng.beta`), como acima. O notebook do curso sorteia de uma Normal e corrige com pesos (amostragem por importância): o resultado é o mesmo, mas a conta direta é mais simples.

⚠️ O Bayesiano **não resolve tudo**: enquanto o teste roda, metade do tráfego continua indo para a versão pior. Esse é o problema que o MAB resolve (Aula 35).

## 5 - Ciclo 04: MAB (Multi-Armed Bandit)

📖 **O que é:** um jogador num cassino com várias máquinas caça-níquel ("bandidos de um braço só"), cada uma com uma chance **desconhecida** de pagar. Ele quer **ganhar o máximo** no total, não só descobrir a melhor máquina (Aula 36). Aplicações: recomendações, anúncios e testes A/B em que perder conversão custa caro.

![Explorar × aproveitar](imagens/ab_explorar_aproveitar.svg)

🧠 **O dilema explorar × aproveitar (Aula 39)**
- Explorar demais: mais tempo testando, menos tempo ganhando.
- Aproveitar cedo demais: não houve tempo para conhecer a verdadeira performance, e a outra opção podia ser melhor.

![A/B fixo × MAB](imagens/ab_alocacao_mab.svg)

🔎 Na figura (A = 5%, B = 10%), o Thompson já manda **98%** dos visitantes para B no fim, enquanto o teste A/B mantém 50% na versão pior até o último visitante.

🧠 **As políticas (agentes) do curso**

| Agente | Como escolhe | Ponto forte | Ponto fraco |
|---|---|---|---|
| **Aleatório** (Aula 37) | Sempre ao acaso, em partes iguais | É o próprio teste A/B/n: referência mínima | Não aproveita nada do que aprende |
| **Onisciente** (Aula 38) | Já sabe qual é o melhor | Referência do máximo possível | Não existe na prática |
| **Guloso** (Aula 40) | Testa cada braço e fica com o melhor até agora | Simples | Decide cedo demais e pode ficar preso no errado |
| **ε-Guloso** (Aula 41) | Com probabilidade ε explora ao acaso; senão, aproveita o melhor | Equilibra os dois | A dificuldade é escolher o ε |
| **UCB** (Aula 36) | Escolhe o maior "limite superior de confiança" | Explora o que tem mais incerteza | Citado no curso, não implementado |
| **Thompson** (Aula 42) | Sorteia uma taxa de cada Beta e escolhe o maior sorteio | Explora quando há dúvida e aproveita quando há certeza, sozinho | Exige a recompensa rapidamente para atualizar |

![ε decrescente](imagens/ab_epsilon_decrescente.svg)

🔎 Com **ε decrescente** de 1% por rodada, ε = (1 − 0,01)^rodada: começa em 100%, cai para **37%** depois de 100 rodadas e quase zera no fim. Resolve o problema do ε fixo, que continua explorando 10% para sempre.

![Thompson Sampling](imagens/ab_thompson.svg)

🧠 **Por que o Thompson funciona (Aula 42):** no começo as Betas se **sobrepõem** e os sorteios ora favorecem A, ora B (exploração). Com os dados, as curvas **se separam**, e o sorteio de B quase sempre ganha (aproveitamento). Não há parâmetro para ajustar.

![Agentes do MAB](imagens/ab_agentes_mab.svg)

🔎 A tabela da Aula 43 mostra o padrão: com braços de 25% e 80%, o Aleatório (o A/B) faz **525** vendas, o Thompson **795** e o Onisciente **800**. Quando os braços ficam próximos (25% × 27%), o ε-Guloso cai para **200** e o Thompson continua em **268**, perto dos **270** do Onisciente.

```python
import numpy as np
rng = np.random.default_rng(42)
sucessos, fracassos = np.zeros(k), np.zeros(k)      # um par por braço (versão, segmento, anúncio)

def escolher_thompson():
    return int(np.argmax(rng.beta(1 + sucessos, 1 + fracassos)))

def atualizar(braco, converteu):
    sucessos[braco] += converteu
    fracassos[braco] += 1 - converteu
```

✅ **O que fazer**
- Medir os agentes pela **recompensa total** e pelo **arrependimento** (*regret*): quanto se ganhou a menos que o Onisciente.
- Repetir a simulação em vários **episódios** (o curso usa 1.000 rodadas e 200 episódios) para ver o comportamento médio, e não a sorte de uma rodada.

⚠️ O MAB **não dá um p-valor** nem mede bem a diferença entre as versões ruins, porque quase não as testa. Para provar o ganho, mantenha um **grupo de controle fixo** (Fase 05).

# FASE 04: AVALIAÇÃO

📖 **O que é:** o passo 5 do curso, **tirar as conclusões**: traduzir o resultado para o negócio e conferir se o método escolhido foi o ideal.

## 1 - Traduzindo o resultado para o negócio

✅ **O que fazer**
- Dizer a **decisão** numa frase: "implantar B", "manter A" ou "inconclusivo: testar de novo com X visitantes".
- Converter em **R$**: diferença de conversão × visitas por mês × ticket × margem, com o **intervalo** (o melhor e o pior caso plausíveis).
- Comparar com o **custo** da mudança (Aula 8): um efeito real, mas menor que o custo, não vale a implantação.

⚠️ Significância estatística não é relevância de negócio. Com muitos dados, uma diferença de 0,01 ponto vira "significativa" e não paga nada.

## 2 - Como escolher o método ideal

📖 Não existe um método melhor em tudo. Cada um troca **rigor**, **velocidade** e **custo do aprendizado** de um jeito:

| Critério | A/B | A/B/n | Bayesiano | MAB (Thompson) |
|---|---|---|---|---|
| Versões | 2 | 3 ou mais | 2 ou mais | 2 ou mais |
| Amostra | Fixa, calculada antes | Fixa, maior (correção) | Flexível (regra de parada) | Contínua |
| Pode olhar todo dia? | ❌ (invalida o α) | ❌ | ✅ | ✅ |
| Resultado | p-valor e IC | p-valor geral + post-hoc | P(B > A) e perda esperada | Recompensa total |
| Tráfego na versão pior | 50% até o fim | 1 − 1/k até o fim | Igual ao A/B até parar | Cai sozinho |
| Mede bem o tamanho do efeito? | ✅ | ✅ | ✅ | ⚠️ só da melhor |
| Melhor para | Decisão única e auditável | Várias ideias de uma vez | Decidir rápido, falar em chance | Otimizar sempre (anúncios, campanhas, recomendações) |

✅ **Checklist de escolha**
1. Preciso de um **p-valor** para um relatório ou auditoria? → **frequentista** (A/B ou A/B/n).
2. Tenho **3 ou mais** versões? → **A/B/n** com correção, ou MAB.
3. O negócio quer resposta **antes** da amostra completa, ou em "chance de B ser melhor"? → **Bayesiano**.
4. Cada visitante na versão pior **custa caro**, ou o teste **se repete** sempre? → **MAB (Thompson)** com grupo de controle.
5. A amostra necessária é **inviável**? → aumentar o efeito mínimo, usar o Bayesiano ou o MAB.

💡 **A combinação mais comum:** validar uma vez com A/B/n (lendo também P(B > A) e a perda esperada) e operar no dia a dia com MAB + controle fixo. Foi a escolha do projeto de Segmentação do portfólio (veja **💼 No portfólio**).

## 3 - Erros comuns em testes A/B

| Erro | O que acontece | Como evitar |
|---|---|---|
| **Espiar e parar cedo** | Falsos vencedores muito acima dos 5% (27,2% na simulação do Ciclo 03) | Amostra fixa, ou Bayesiano com regra de parada |
| **Várias comparações sem correção** | 40% de chance de falso vencedor com 10 pares | Teste geral + post-hoc com Bonferroni |
| **Sorteio quebrado (SRM)** | Grupos diferentes antes do teste | Qui-quadrado dos tamanhos; investigar se p < 0,01 |
| **Mudar mais de uma coisa** | Não se sabe o que causou o efeito | Uma mudança por variante (Aula 15) |
| **Teste curto** | Efeito novidade e dias da semana distorcem | Pelo menos 1 semana cheia |
| **Métrica errada** | Mais cliques e menos vendas | Métrica principal ligada a R$ + métricas de proteção |
| **Tabela errada no qui-quadrado** | Convertidos × total em vez de convertidos × não convertidos | Montar a tabela com quem converteu e quem **não** converteu |
| **Ler p > 0,05 como "iguais"** | Descartar uma ideia boa por falta de amostra | Olhar o IC e o poder |

# FASE 05: IMPLANTAÇÃO

📖 **O que é:** colocar a decisão em produção e, quando o teste vira rotina, **automatizar** a alocação e o acompanhamento.

## 1 - Teste em produção: bandit + grupo de controle

![Produção](imagens/ab_producao.svg)

| Peça | Como fazer |
|---|---|
| **Grupo de controle fixo (~10%)** | Divisão igual entre as versões. Sem ele, não dá para provar quanto o bandit está ganhando. |
| **Estado do bandit** | Uma tabela com sucessos e tentativas por versão. Pode ser uma **API Flask** com um endpoint que recebe cada conversão e atualiza os dados (Aulas 44 e 45) ou um job diário. |
| **Limites de segurança** | Uma fração mínima para cada versão (para continuar aprendendo), alertas e um teto de exposição. |
| **Relatório** | Semanal: P(B > A), perda esperada, P(dá lucro) e ganho contra o controle. |

## 2 - Monitoramento e quando refazer o teste

✅ **O que fazer**
- Acompanhar a métrica principal **contra o controle** ao longo do tempo: o efeito de uma novidade costuma diminuir.
- **Refazer o teste** quando mudar a oferta, o preço, o custo ou o público, ou quando o relatório mostrar uma mudança de comportamento.
- Registrar cada teste (hipótese, datas, amostra, resultado e decisão) num **histórico de experimentos**: evita repetir testes e mostra o que funciona.

# ✅ RESPOSTA À PERGUNTA DE NEGÓCIO

📖 Responda à pergunta da Fase 01 numa frase, com o número principal e o intervalo, e diga **qual método foi usado e por quê**. Exemplo: "a página B aumenta a conversão em 1,8 ponto (IC 95%: 0,9 a 2,7), cerca de R$ X por mês; validado com um A/B frequentista de 4.720 visitantes por grupo".

# 💡 INSIGHTS

📖 O que o teste ensinou além da decisão: segmentos que reagiram diferente, métricas de proteção que mudaram, ideias para o próximo teste.

# 🚀 PRÓXIMOS PASSOS

📖 Melhorias em ordem de impacto: o próximo teste, a automação (MAB + controle), o painel de acompanhamento.

# 💼 No portfólio

**Segmentação de clientes — `Notebooks/Teste A-B da campanha.ipynb`** aplica os 4 ciclos deste guia com as respostas reais de uma campanha (custo de 3 por contato, receita de 11 por resposta, ponto de equilíbrio de 27,3%):

| Ciclo | Resultado |
|---|---|
| 01 · A/B | Na taxa de resposta, não deu para provar (19,2% × 16,3%, p = 0,145; seriam precisos 903 contatos por grupo). No lucro por cliente, sim (Welch, p < 0,001) |
| 02 · A/B/n | 4 regras de contato, 134 contatos cada: só "só Premium" passou do ponto de equilíbrio (34,2%) e foi a única diferente depois da correção de Bonferroni |
| 03 · Bayesiano | O p-valor diário cruzou 5% e voltou a subir; o Bayesiano decidiu no dia 3 pela perda esperada |
| 04 · MAB | Em 1.000 contatos, o Thompson teve 316 respostas e lucro de +480, contra 164 e −1.200 da divisão em partes iguais |

**Decisão:** A/B/n para validar a regra + Thompson Sampling em produção, com controle fixo de ~10%.

# 📚 Pontos de atenção no material do curso

Ao conferir as aulas e os notebooks do curso, alguns trechos merecem cuidado. O guia segue a forma correta:

| Onde | O que diz | O correto |
|---|---|---|
| Notebook *PA James ABN* | Calcula a amostra do qui-quadrado com `TTestIndPower` e marca o `GofChisquarePower` como "aplicação errada" | O `GofChisquarePower` é o feito para o qui-quadrado (w de Cohen); o `TTestIndPower` superestima a amostra (Fase 03, item 3) |
| Notebook *PA James* | A coluna `non_converted` da tabela do qui-quadrado recebe o **total** de visitantes | A tabela usa convertidos e **não** convertidos (total − convertidos) |
| Resumo "Tipos de testes" do *PA James ABN* | "T Test (n > 30) / Z Test (n < 30)" | O critério é o tipo de métrica: Z para proporções, t para médias com desvio estimado (Aula 22 e mapa da Aula 9). E, pela regra clássica, n pequeno pede o t, não o Z |
| Aula 22, teste Z de 2 proporções | "Se o p-valor é **menor** do que o nível de significância, **não** há evidências…" | Invertido: p **menor** que α → **há** evidência de diferença (rejeita H0) |
| Aula 11, problema 2, solução 1 | Teste: "One-Sample T-test" para 2 grupos | Two-Sample t-test, como na Aula 10 |
| Exercício 8 (notebook de exercícios) | H0: "o número de clientes **não** é igual todos os dias" | H0 é o "sem efeito": o número **é** igual todos os dias; H1: não é |
| Aula 33, passo 4.2 | "Priori: distribuição Binomial; Posteriori: Beta" | A **priori** é a Beta(1, 1); a **Binomial** é a verossimilhança (os dados); a **posteriori** é a Beta atualizada |

# 📚 Glossário

Termos e siglas deste guia, em ordem alfabética.

| Termo | O que significa |
|---|---|
| **A priori / a posteriori** | No Bayesiano: o que se acredita antes dos dados (Beta(1, 1)) e depois de atualizar com eles. |
| **A/A (teste)** | Teste com duas versões idênticas; serve para conferir o sorteio e mostrar falsos positivos. |
| **A/B/n** | Teste A/B com 3 ou mais versões. |
| **Arrependimento (regret)** | Quanto um agente de MAB ganhou a menos que o Onisciente. |
| **Bicaudal / unicaudal** | H1 "diferente" (as duas caudas) ou "maior/menor" (uma cauda só). |
| **Bonferroni** | Correção para várias comparações: multiplica cada p-valor pelo número de comparações. |
| **Braço (arm)** | Cada opção que o MAB pode escolher (versão, anúncio, segmento). |
| **CTR** | *Click-through rate*: cliques ÷ visualizações. |
| **Distribuição Beta** | Distribuição de uma taxa entre 0 e 1; Beta(1 + sucessos, 1 + fracassos). |
| **Efeito mínimo (effect size)** | A menor diferença que vale a pena enxergar; define o tamanho da amostra. |
| **Erro α acumulado** | α* = 1 − (1 − α)ⁿ: chance de pelo menos um falso positivo em n comparações. |
| **Erro tipo I / tipo II** | Tipo I (α): ver efeito que não existe. Tipo II (β): deixar passar um efeito real. |
| **ε-Guloso** | Agente que explora ao acaso com probabilidade ε e aproveita o melhor no resto do tempo. |
| **Explorar × aproveitar** | Testar opções para aprender × usar a melhor conhecida. |
| **Flag do teste** | Coluna que registra o grupo de cada usuário. |
| **h de Cohen** | Tamanho do efeito para duas proporções. |
| **Intervalo de confiança (IC)** | Faixa de valores plausíveis; 95% dos intervalos construídos assim contêm o valor real. |
| **MAB** | *Multi-Armed Bandit*: desloca o tráfego para a melhor opção enquanto aprende. |
| **Monte Carlo** | Sortear milhares de valores para calcular uma probabilidade ou uma média. |
| **Nível de confiança** | 1 − α; em geral 95%. |
| **Nível de significância (α)** | Risco máximo de rejeitar H0 sendo ela verdadeira; em geral 5%. |
| **p-valor** | Chance de ver um resultado tão extremo quanto o observado se H0 fosse verdade. |
| **Perda esperada** | No Bayesiano, quanto se perde em média ao escolher uma versão se ela for a pior. |
| **Perda máxima aceitável** | Limite da perda esperada para decidir; definido no design. |
| **Poder (1 − β)** | Chance de o teste enxergar um efeito que existe; em geral 80%. |
| **Post-hoc** | Comparações par a par, corrigidas, depois de um teste geral significativo. |
| **SRM** | *Sample ratio mismatch*: os grupos ficaram com tamanhos diferentes do planejado. |
| **Suposições paramétricas** | Independência, normalidade e variâncias iguais, exigidas pelos testes t e ANOVA. |
| **Thompson Sampling** | Agente de MAB que sorteia uma taxa de cada Beta e escolhe o maior sorteio. |
| **Verossimilhança** | A chance de observar os dados para cada valor possível da taxa (aqui, a Binomial). |
| **w de Cohen** | Tamanho do efeito do qui-quadrado (3 ou mais proporções). |